# 3. Metrics & Calibration

This notebook tests the robustness of governance decisions under three forms of uncertainty: stochastic variation within confidence intervals (Monte Carlo), systematic feature perturbation, and methodological variation (dataset and mode sensitivity).

In [ ]:
# Configuration
import json, sys, csv, random, copy
from pathlib import Path

def _repo_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config" / "harness_settings.json").is_file():
            return candidate
    return start

REPO_ROOT = _repo_root()
sys.path.insert(0, str(REPO_ROOT / 'engine'))
import corrected_public_engine_v1_1 as engine

CANONICAL_PATH = REPO_ROOT / 'data' / 'canonical' / 'canonical_dataset.json'
NORMALISED_PATH = REPO_ROOT / 'data' / 'canonical' / 'public_normalised_dataset.json'
PERTURBATION_PATH = REPO_ROOT / 'data' / 'canonical' / 'perturbation_dataset.json'

MC_ITERATIONS = 200
MC_SEED = 42
PROFILES = ['permissive', 'moderate', 'strict', 'very_strict']

with open(CANONICAL_PATH) as f:
    canonical = json.load(f)
with open(NORMALISED_PATH) as f:
    normalised = json.load(f)
with open(PERTURBATION_PATH) as f:
    perturbation = json.load(f)

orig12 = sorted(canonical['cases'].keys())
metrics = []  # collect all metrics for CSV

## 3.1 Monte Carlo Stability

Each feature encoding carries confidence intervals [value_low, value_high]. By uniformly sampling within these bounds across 200 iterations, we test whether governance decisions are stable under the uncertainty inherent in the evidence base.

In [ ]:
random.seed(MC_SEED)

stable_count = 0
for cid in orig12:
    case_feats = canonical['cases'][cid]['features']
    # Deterministic verdict
    flat = {k: v['value_primary'] for k, v in case_feats.items()}
    det = engine.evaluate_case({'case_id': cid, 'features': flat},
                               profile_name='moderate', mode=engine.MODE_REPLAY)
    det_verdict = det['approved']
    
    # Monte Carlo
    all_match = True
    for _ in range(MC_ITERATIONS):
        sampled = {}
        for fname, fobj in case_feats.items():
            lo = fobj.get('value_low', fobj['value_primary'])
            hi = fobj.get('value_high', fobj['value_primary'])
            sampled[fname] = random.uniform(lo, hi)
        mc_r = engine.evaluate_case({'case_id': cid, 'features': sampled},
                                    profile_name='moderate', mode=engine.MODE_REPLAY)
        if mc_r['approved'] != det_verdict:
            all_match = False
            break
    
    if all_match:
        stable_count += 1

print(f'Monte Carlo stability: {stable_count}/12 cases stable across {MC_ITERATIONS} iterations')
assert stable_count == 12, f'Q29: Expected 12, got {stable_count}'
print('Q29: 12/12 stable ✓')
metrics.append(('monte_carlo', 'cases_stable', 12, 'Q29', 'PASS'))
metrics.append(('monte_carlo', 'iterations', MC_ITERATIONS, 'Q29', 'PASS'))

## 3.2 Perturbation Analysis

The perturbation dataset applies simultaneous adversarial ±0.05 shifts to three features (uncertainty_calibration, bias_harm_index, traceability_integrity). Comparing governance verdicts between canonical and perturbed datasets across 48 case-profile configurations (12 cases × 4 profiles) reveals stability under this worst-case envelope.

In [ ]:
# Load perturbation dataset and compare against canonical
with open(PERTURBATION_PATH) as f:
    perturbation = json.load(f)

verdict_flips = 0
gate_flips = 0
moderate_flips = 0
case_profile_stable = 0
case_profile_total = 0
flip_details = []

gate_names = ['gate_safety', 'gate_evidence', 'gate_bias', 'gate_calibration', 'gate_traceability']

for prof in PROFILES:
    for cid in orig12:
        case_profile_total += 1
        # Canonical verdict
        flat_c = {k: v['value_primary'] for k, v in canonical['cases'][cid]['features'].items()}
        r_c = engine.evaluate_case({'case_id': cid, 'features': flat_c},
                                   profile_name=prof, mode=engine.MODE_REPLAY)
        # Perturbed verdict
        flat_p = {k: v['value_primary'] for k, v in perturbation['cases'][cid]['features'].items()}
        r_p = engine.evaluate_case({'case_id': cid, 'features': flat_p},
                                   profile_name=prof, mode=engine.MODE_REPLAY)
        
        if r_c['approved'] != r_p['approved']:
            verdict_flips += 1
            flip_details.append(f'{cid}/{prof}')
            if prof == 'moderate':
                moderate_flips += 1
        else:
            case_profile_stable += 1
        
        for g in gate_names:
            if r_c.get(g, 1) != r_p.get(g, 1):
                gate_flips += 1

print(f'Perturbation: {case_profile_stable}/{case_profile_total} case-profile pairs stable')
print(f'Verdict flips: {verdict_flips} ({flip_details})')
print(f'Gate flips: {gate_flips}')
print(f'Moderate flips: {moderate_flips}')

assert case_profile_stable == 46, f'Q30: Expected 46, got {case_profile_stable}'
assert verdict_flips == 2, f'Q31: Expected 2 flips, got {verdict_flips}'
assert gate_flips == 10, f'Q32: Expected 10 gate flips, got {gate_flips}'
assert moderate_flips == 0, f'Q33: Expected 0 moderate flips, got {moderate_flips}'
print(f'\nQ30: 46/48 stable ✓, Q31: 2 flips ✓, Q32: 10 gate flips ✓, Q33: 0 moderate flips ✓')

metrics.append(('perturbation', 'verdicts_stable', case_profile_stable, 'Q30', 'PASS'))
metrics.append(('perturbation', 'verdict_flips', verdict_flips, 'Q31', 'PASS'))
metrics.append(('perturbation', 'gate_flips', gate_flips, 'Q32', 'PASS'))
metrics.append(('perturbation', 'moderate_flips', moderate_flips, 'Q33', 'PASS'))

## 3.3 Google DR Sensitivity

As the sole approved case under the moderate profile, Google DR's decision boundary warrants focused sensitivity analysis.

In [ ]:
# Google DR sensitivity: perturb each feature across +-0.20 range
gdr_feats = {k: v['value_primary'] for k, v in canonical['cases']['google_dr']['features'].items()}
gdr_base = engine.evaluate_case({'case_id': 'google_dr', 'features': gdr_feats},
                                profile_name='moderate', mode=engine.MODE_REPLAY)

flip_points = 0
flip_features = set()
for fname in gdr_feats:
    for delta in [-0.20, -0.15, -0.10, -0.05, +0.05, +0.10, +0.15, +0.20]:
        test = dict(gdr_feats)
        test[fname] = max(0, min(1, gdr_feats[fname] + delta))
        r = engine.evaluate_case({'case_id': 'google_dr', 'features': test},
                                 profile_name='moderate', mode=engine.MODE_REPLAY)
        if r['approved'] != gdr_base['approved']:
            flip_points += 1
            flip_features.add(fname)

print(f'Google DR flip points: {flip_points} across {len(flip_features)} features')
# Q14: Manuscript reports 8 flip points; engine produces 7 at ±0.20/0.05 steps.
# The 8th is a floating-point boundary case (evidence_strength 0.70-0.20=0.4999...94
# vs exact 0.50 threshold). Annotated, does not affect any governance decision.
assert flip_points == 7, f'Q14: Expected 7 flip points, got {flip_points}'
print('Q14: 7 flip points ✓ (manuscript reports 8; see N14 annotation)')
metrics.append(('google_dr', 'flip_points', flip_points, 'Q14', 'PASS'))

## 3.4 Dual Dataset Invariance

The canonical and normalised datasets encode the same 12 cases with different serialisation. Running both through all four profiles produces 480 field comparisons. Perfect invariance confirms that governance decisions are encoding-independent.

In [ ]:
fields_compared = 0
divergences = 0
compare_fields = ['approved', 'safety_gate', 'evidence_gate', 'bias_gate',
                  'calibration_gate', 'traceability_gate', 'governance_outcome',
                  'compensatory_score', 'compensatory_approved', 'compensatory_threshold']

for prof in PROFILES:
    for cid in orig12:
        flat_c = {k: v['value_primary'] for k, v in canonical['cases'][cid]['features'].items()}
        flat_n = {k: v['value_primary'] for k, v in normalised['cases'][cid]['features'].items()}
        
        r_c = engine.evaluate_case({'case_id': cid, 'features': flat_c},
                                   profile_name=prof, mode=engine.MODE_REPLAY)
        r_n = engine.evaluate_case({'case_id': cid, 'features': flat_n},
                                   profile_name=prof, mode=engine.MODE_REPLAY)
        
        for field in compare_fields:
            fields_compared += 1
            v_c = r_c.get(field)
            v_n = r_n.get(field)
            if isinstance(v_c, float) and isinstance(v_n, float):
                if abs(v_c - v_n) > 1e-10:
                    divergences += 1
            elif v_c != v_n:
                divergences += 1

print(f'Dual dataset invariance: {fields_compared} fields compared, {divergences} divergences')
assert fields_compared == 480, f'Q37: Expected 480, got {fields_compared}'
assert divergences == 0, f'Q38: Expected 0 divergences, got {divergences}'
print('Q37: 480/480 ✓, Q38: 0 divergences ✓')
metrics.append(('invariance', 'fields_matched', fields_compared, 'Q37', 'PASS'))
metrics.append(('invariance', 'divergences', divergences, 'Q38', 'PASS'))

## 3.5 Mode Sensitivity

The engine supports two execution modes: replay (gate-only) and canonical_full (gates + SCM abstention + fallback). Under the moderate profile, mode choice has zero effect.

In [ ]:
moderate_divergences = 0
permissive_flips = 0
permissive_flip_cases = []

for prof in PROFILES:
    for cid in orig12:
        flat = {k: v['value_primary'] for k, v in canonical['cases'][cid]['features'].items()}
        r_replay = engine.evaluate_case({'case_id': cid, 'features': flat},
                                        profile_name=prof, mode=engine.MODE_REPLAY)
        r_full = engine.evaluate_case({'case_id': cid, 'features': flat},
                                      profile_name=prof, mode=engine.MODE_CANONICAL_FULL)
        
        if r_replay['approved'] != r_full['approved']:
            if prof == 'moderate':
                moderate_divergences += 1
            elif prof == 'permissive':
                permissive_flips += 1
                permissive_flip_cases.append(cid)

print(f'Mode sensitivity:')
print(f'  Moderate divergences: {moderate_divergences}')
print(f'  Permissive flips: {permissive_flips} ({permissive_flip_cases})')

assert moderate_divergences == 0, f'Q39: Expected 0, got {moderate_divergences}'
assert permissive_flips == 3, f'Q40: Expected 3, got {permissive_flips}'
print('\nQ39: 0 moderate divergences ✓, Q40: 3 permissive flips ✓')
metrics.append(('mode_sensitivity', 'moderate_divergences', moderate_divergences, 'Q39', 'PASS'))
metrics.append(('mode_sensitivity', 'permissive_flips', permissive_flips, 'Q40', 'PASS'))

## 3.6 Write Outputs

In [ ]:
# metrics_summary.csv
ms_path = REPO_ROOT / 'outputs' / 'tables' / 'metrics_summary.csv'
with open(ms_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['test', 'metric', 'value', 'rtm_target', 'status'])
    for row in metrics:
        w.writerow(row)
print(f'Written: {ms_path}')

# calibration_summary.txt
cs_path = REPO_ROOT / 'outputs' / 'figures' / 'calibration_summary.txt'
with open(cs_path, 'w') as f:
    f.write('Monte Carlo Stability Summary\n')
    f.write(f'Iterations: {MC_ITERATIONS}, Seed: {MC_SEED}\n')
    f.write(f'Cases stable: {stable_count}/12 (100%)\n')
    f.write(f'Perturbation: {case_profile_stable}/48 case-profile pairs stable (95.8%)\n')
    f.write(f'Google DR flip points: {flip_points}\n')
    f.write(f'Dual dataset invariance: {fields_compared}/480 fields matched\n')
    f.write(f'Mode sensitivity (moderate): {moderate_divergences} divergences\n')
print(f'Written: {cs_path}')